In [1]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "data").exists():
    raise FileNotFoundError("Open this notebook from the NutriCore repository or its notebooks directory.")

NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))
os.chdir(PROJECT_ROOT)

# Hybrid Recommendation System

## Project Overview

The previous notebooks developed individual components of the recipe recommendation pipeline, each capturing a different aspect of user preference and recipe quality. While each approach performs well independently, relying on a single recommendation strategy can limit the overall recommendation quality.

This notebook integrates the previously developed components into a unified Hybrid Recommendation System capable of generating more accurate and personalized recipe recommendations.

The hybrid model combines:

- Nutrition Matching
- Popularity-Based Ranking
- Content-Based Similarity
- Collaborative Filtering

Each recommendation component contributes a normalized score, which is then combined to produce a final ranking of recipes.

### Notebook Objectives

This notebook:

- Load the processed datasets and previously trained models.
- Generate recommendation scores from each individual component.
- Normalize scores onto a common scale.
- Combine the normalized scores using a weighted fusion strategy.
- Generate final hybrid recommendations.
- Compare individual recommendation methods with the final hybrid model.

This notebook represents the complete recommendation pipeline developed throughout the project and serves as the final recommendation engine for the application.

In [2]:
# Import Libraries

import joblib
from scipy.sparse import load_npz
import numpy as np
import pandas as pd
from nb_utils.recommender_nb import *
from nb_utils.nutrition_nb import *

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

## Load Datasets and Models

The hybrid recommendation system combines multiple recommendation techniques developed in previous notebooks. This requires loading the processed datasets together with the trained models and saved artifacts generated throughout the project.

The following resources are loaded:

- Processed recipe dataset
- Processed review dataset
- Recipe popularity dataset
- Content-based recommendation artifacts
- Collaborative filtering models

Loading these resources ensures that every recommendation component uses the same processed data and previously trained models.

In [3]:
# Load Datasets

recipes_df = load_dataset("data/processed/recipes_processed.parquet")
print(f"Recipes Shape    : {recipes_df.shape}")

reviews_df = load_dataset("data/processed/reviews_processed.parquet")
print(f"Reviews Shape    : {reviews_df.shape}")

popularity_df = load_dataset("data/processed/recipe_popularity.parquet")
print(f"Popularity Shape : {popularity_df.shape}")


Recipes Shape    : (302965, 16)
Reviews Shape    : (788843, 8)
Popularity Shape : (155842, 5)


## Load Recommendation Models

The hybrid recommendation system reuses the trained models developed in previous notebooks. Loading these saved artifacts avoids recomputing computationally expensive operations and ensures consistency across the recommendation pipeline.

The following artifacts are loaded:

- TF-IDF Vectorizer
- TF-IDF Matrix
- SVD Collaborative Filtering Model

These models are used to generate the content-based and collaborative filtering scores required by the hybrid recommendation system.

In [4]:
# Load Recommendation Models

tfidf_vectorizer = joblib.load("models/tfidf_vectorizer.pkl")
print("✓ TF-IDF Vectorizer loaded successfully.")

tfidf_matrix = load_npz("models/tfidf_matrix.npz")
print("✓ TF-IDF Matrix loaded successfully.")

svd_model = joblib.load("models/svd_model.pkl")
print("✓ SVD Model loaded successfully.")


✓ TF-IDF Vectorizer loaded successfully.
✓ TF-IDF Matrix loaded successfully.
✓ SVD Model loaded successfully.


## Recommendation Components

The hybrid recommendation system combines multiple recommendation techniques, with each component capturing a different aspect of recipe relevance and user preference.

The recommendation components are:

- **Nutrition Matching:** Recommends recipes that closely satisfy the user's nutritional requirements.
- **Popularity Ranking:** Prioritizes recipes that are consistently well-rated by the community.
- **Content-Based Similarity:** Recommends recipes similar to those the user has previously liked.
- **Collaborative Filtering (SVD):** Learns latent user and recipe preferences from historical ratings to generate personalized recommendations.

Each component produces an independent recommendation score. These scores will be normalized to a common scale before being combined to generate the final hybrid recommendation score.

In [5]:
# Create Helper Mappings

recipe_id_to_index = pd.Series(
    recipes_df.index,
    index=recipes_df["RecipeId"]
).to_dict()

index_to_recipe_id = pd.Series(
    recipes_df["RecipeId"].values,
    index=recipes_df.index
).to_dict()

print(f"Indexed Recipes : {len(recipe_id_to_index):,}")

Indexed Recipes : 302,965


## Hybrid Scoring Strategy

The hybrid recommendation system combines multiple independent
scoring mechanisms into a unified ranking.

### Components

1. Nutrition Score (Dynamic)
   - Based on macro alignment
   - Computed per meal

2. Content Score (Static per user)
   - Based on TF-IDF similarity
   - Personalized using liked recipes

3. Popularity Score (Static)
   - Based on aggregated ratings

4. Collaborative Score (SVD)
   - Learns user-item interaction patterns

### Final Score

The final hybrid score is computed as:

FinalScore =
0.45 * NutritionScore +
0.30 * PopularityScore +
0.20 * ContentScore +
0.05 * SVDScore

All scores are normalized to the range [0, 1].

In [6]:
# Define Target User

USER_ID = 1533

In [7]:
# Content-Based Scores

content_df = compute_content_recommendations(
    user_id=USER_ID,
    recipes_df=recipes_df,
    reviews_df=reviews_df,
    recipe_id_to_index=recipe_id_to_index,
    tfidf_matrix=tfidf_matrix,
)

content_df = content_df[["RecipeId", "ContentScore"]]

In [8]:
# SVD Scores

svd_df = recommend_svd(
    user_id=USER_ID,
    model=svd_model,
    reviews_df=reviews_df,
    recipes_df=recipes_df,
)

svd_df = svd_df[["RecipeId", "SVDScore"]]

In [9]:
# Popularity Scores

popularity_df = popularity_df[
    ["RecipeId", "PopularityScore"]
]

In [10]:
# Merge Static Scores

hybrid_static_df = content_df.merge(
    popularity_df,
    on="RecipeId",
    how="left"
)

hybrid_static_df = hybrid_static_df.merge(
    svd_df,
    on="RecipeId",
    how="left"
)



## Handling Missing Scores

### SVDScore
Missing values are filled with **0** because:
- No interaction → no collaborative signal
- SVD has **low weight (0.05)**, so impact is minimal
- Scores are **positively skewed**, most recipes naturally have low SVD relevance

### PopularityScore
Missing values are filled with the **mean** because:
- No reviews = unknown, not bad
- Popularity has **high weight (0.30)**, so 0 would unfairly penalize new recipes
- Mean acts as a **neutral baseline**

### Summary
- SVD → 0 (no signal, low impact)
- Popularity → mean (neutral, high impact)

In [11]:
# Fill Missing Values (Final Strategy)

# Mean popularity score
mean_popularity = hybrid_static_df["PopularityScore"].mean()

# Fill NaNs
hybrid_static_df["SVDScore"] = hybrid_static_df["SVDScore"].fillna(0)
hybrid_static_df["PopularityScore"] = hybrid_static_df["PopularityScore"].fillna(mean_popularity)

## Score Validation

Before computing the final hybrid score, we validate that:

1. There are **no missing values (NaNs)** in:
   - ContentScore
   - PopularityScore
   - SVDScore

2. All scores are properly **normalized in the range [0, 1]**



In [12]:
# Check for NaNs

content_nan = hybrid_static_df["ContentScore"].isna().any()
popularity_nan = hybrid_static_df["PopularityScore"].isna().any()
svd_nan = hybrid_static_df["SVDScore"].isna().any()

print("ContentScore has NaN:", content_nan)
print("PopularityScore has NaN:", popularity_nan)
print("SVDScore has NaN:", svd_nan)


# Check normalization (0–1 range)

content_range = hybrid_static_df["ContentScore"].between(0, 1).all()
popularity_range = hybrid_static_df["PopularityScore"].between(0, 1).all()
svd_range = hybrid_static_df["SVDScore"].between(0, 1).all()

print("ContentScore in [0,1]:", content_range)
print("PopularityScore in [0,1]:", popularity_range)
print("SVDScore in [0,1]:", svd_range)

ContentScore has NaN: False
PopularityScore has NaN: False
SVDScore has NaN: False
ContentScore in [0,1]: True
PopularityScore in [0,1]: True
SVDScore in [0,1]: True


## Final Hybrid Score Computation

After validating that all component scores are:
- Free from missing values
- Properly normalized in the range [0, 1]

We compute the final hybrid score using a weighted combination of all components.

### Final Score Formula

FinalScore =
0.45 * NutritionScore +
0.30 * PopularityScore +
0.20 * ContentScore +
0.05 * SVDScore

At this stage, only the **static components** are combined:
- ContentScore
- PopularityScore
- SVDScore

The NutritionScore will be incorporated dynamically during meal planning.

In [13]:
# Compute Static Hybrid Score

hybrid_static_df["StaticScore"] = (
    0.30 * hybrid_static_df["PopularityScore"] +
    0.20 * hybrid_static_df["ContentScore"] +
    0.05 * hybrid_static_df["SVDScore"]
)

hybrid_static_df = hybrid_static_df.sort_values(
    by="StaticScore",
    ascending=False
).reset_index(drop=True)

hybrid_static_df.head()

,RecipeId,ContentScore,PopularityScore,SVDScore,StaticScore
0,4843.0,0.790826,0.8994,0.935080,0.474739
1,50847.0,0.642056,0.9677,0.999095,0.468676
2,26632.0,0.640586,0.9498,0.989849,0.462550
3,391431.0,0.538905,0.9657,0.989155,0.446949
4,435989.0,0.755520,0.8972,0.496327,0.445080


## Integration with Nutrition-Based Recommendations

The final hybrid recommendation is computed dynamically by combining:

- StaticScore (precomputed)
- NutritionScore (computed per query)

This ensures that:
- User preferences are captured via static components
- Nutritional goals are satisfied dynamically

This separation improves efficiency by avoiding recomputation of static scores.

In [14]:
# Hybrid Recommendation Function

def generate_hybrid_recommendations(
    recipes_df,
    hybrid_static_df,
    calories,
    protein,
    carbs,
    fat,
    meals_per_day,
    diet_type="Non-Vegetarian",
    top_n=10
):

    # Step 1: Nutrition recommendations
    nutrition_df = generate_recommendations(
        df=recipes_df,
        calories=calories,
        protein=protein,
        carbs=carbs,
        fat=fat,
        meals_per_day=meals_per_day,
        diet_type=diet_type,
    )

    # Step 2: Merge with static scores
    df = nutrition_df.merge(
        hybrid_static_df[["RecipeId", "StaticScore"]],
        on="RecipeId",
        how="left"
    )

    df["StaticScore"] = df["StaticScore"].fillna(0)

    # Step 3: Final score
    df["FinalScore"] = (
        0.45 * df["NutritionScore"] +
        df["StaticScore"]
    )

    df = df.sort_values(
        by="FinalScore",
        ascending=False
    ).reset_index(drop=True)

    if top_n:
        df = df.head(top_n)

    return df

In [15]:
hybrid_recommendations = generate_hybrid_recommendations(
    recipes_df=recipes_df,
    hybrid_static_df=hybrid_static_df,
    calories=2400,
    protein=150,
    carbs=300,
    fat=70,
    meals_per_day=4,
    diet_type="Non-Vegetarian",
    top_n=10
)

hybrid_recommendations

,RecipeId,Name,BestServing,RecommendedCalories,RecommendedProtein,RecommendedCarbs,RecommendedFat,Distance,NutritionScore,StaticScore,FinalScore
0,90772.0,"Black Beans, Chicken and Rice",4,566.10,37.90,61.60,18.10,0.0532,0.9482,0.382396,0.809086
1,72119.0,Shrimp Parmesan,4,573.80,36.93,63.47,18.53,0.0511,0.9502,0.370113,0.797703
2,183021.0,Crock Pot Ground Beef and Beans,8,582.70,38.50,70.60,18.40,0.0358,0.9648,0.359698,0.793858
3,173097.0,Chicken Pizzaiola,4,574.50,39.00,56.00,19.00,0.0796,0.9235,0.377885,0.793460
4,76926.0,Best Baked Ziti,8,584.10,34.20,67.20,19.10,0.0725,0.9301,0.370908,0.789453
5,223628.0,Turkey Panini,4,589.20,36.00,78.70,14.50,0.0545,0.9469,0.362369,0.788474
6,131473.0,Best Ever Chicken Alfredo,4,593.80,31.40,74.70,18.40,0.0765,0.9264,0.370757,0.787637
7,143519.0,Spinach Meatballs,4,526.90,36.50,59.00,15.40,0.0972,0.9074,0.379267,0.787597
8,51751.0,"Angel Hair with Shrimp, Basil, and Toasted Pine Nuts",6,606.50,38.00,62.50,19.20,0.0482,0.9530,0.358441,0.787291
9,535831.0,Quick Chicken and Dumplings,3,592.12,37.12,70.72,17.40,0.0173,0.9828,0.344227,0.786487


## End-to-End Planning Example

### Importing Notebook Modules

We import the nutrition and recommender modules developed in previous stages.

These modules provide:
- Nutrition target calculation
- Content-based recommendations
- SVD-based recommendations
- Nutrition-based scoring

In [16]:
import nb_utils.nutrition_nb as nutrition
import nb_utils.recommender_nb as recommender

import pandas as pd
import numpy as np

In [17]:
user_profile = {
    "age": 22,
    "gender": "male",
    "weight": 77,             # in kg
    "height": 175,            # in cm
    "activity_level": "moderately active",
    "goal": "Moderate Cut"
}

user_profile

{'age': 22,
 'gender': 'male',
 'weight': 77,
 'height': 175,
 'activity_level': 'moderately active',
 'goal': 'Moderate Cut'}

### Basal Metabolic Rate (BMR)

Basal Metabolic Rate represents the number of calories the body
requires at rest to maintain essential physiological functions.

We compute BMR using the Mifflin-St Jeor Equation,
implemented in the nutrition module.

In [18]:
bmr = nutrition.calculate_bmr(
    age=user_profile["age"],
    gender=user_profile["gender"],
    weight=user_profile["weight"],
    height=user_profile["height"]
)

print("BMR:", round(bmr, 2))

BMR: 1758.75


### Total Daily Energy Expenditure (TDEE)

TDEE represents the total number of calories burned per day,
including physical activity.

It is calculated by multiplying BMR with an activity multiplier
based on the user's activity level.

In [19]:
tdee = nutrition.calculate_tdee(
    bmr=bmr,
    activity_level=user_profile["activity_level"]
)

print("TDEE:", round(tdee, 2))

TDEE: 2726.06


### Goal-Based Calorie Targets

Based on TDEE, we generate calorie targets for different fitness goals.

These include:
- Maintenance
- Fat loss variations (Mild, Moderate, Aggressive)
- Muscle gain variations (Lean Bulk, Bulk)

This allows flexible selection based on user preference.

In [20]:
calorie_targets = nutrition.generate_calorie_targets(tdee)

calorie_targets

{'Maintenance': 2726,
 'Mild Cut': 2453,
 'Moderate Cut': 2181,
 'Aggressive Cut': 2045,
 'Lean Bulk': 2862,
 'Bulk': 2999,
 'Aggressive Bulk': 3135}

### Macro Target Generation

From the available calorie targets, we select the user's goal
and compute the corresponding macronutrient distribution.

The macro breakdown is calculated as:

- Protein → based on body weight and goal
- Fat → fixed percentage of total calories (25%)
- Carbohydrates → remaining calories after protein and fat

In [21]:
# Select goal
selected_goal = user_profile["goal"]

# Get calories for that goal
target_calories = calorie_targets[selected_goal]

# Generate macro targets
macro_targets = nutrition.generate_macro_targets(
    weight=user_profile["weight"],
    calories=target_calories,
    goal=selected_goal
)

print("Selected Goal:", selected_goal)
macro_targets

Selected Goal: Moderate Cut


{'Calories': 2181, 'Protein_g': 169.4, 'Fat_g': 60.6, 'Carbs_g': 239.5}

### Macro Validation

We verify that the calculated macronutrients correctly sum up
to the target calorie value.

This ensures consistency between:
- Target calories
- Calories derived from macros

A small tolerance is allowed due to rounding.

In [22]:
validation = nutrition.validate_macros(
    calories=macro_targets["Calories"],
    protein=macro_targets["Protein_g"],
    fat=macro_targets["Fat_g"],
    carbs=macro_targets["Carbs_g"]
)

validation

{'Target Calories': 2181,
 'Macro Calories': 2181.0,
 'Difference': 0.0,
 'Valid': True}

### Macro Quality Assessment

We assess the quality of the macronutrient distribution
based on recommended percentage ranges.

This helps identify:
- Extremely low or high macronutrient proportions
- Potential imbalances in the diet

The assessment provides warnings if any macro falls outside
recommended ranges.

In [23]:
quality = nutrition.assess_macro_quality(
    calories=macro_targets["Calories"],
    protein=macro_targets["Protein_g"],
    fat=macro_targets["Fat_g"],
    carbs=macro_targets["Carbs_g"]
)

quality

{'Protein_%': 31.1,
 'Fat_%': 25.0,
 'Carbohydrates_%': 43.9,
 'Warnings': ['No Warnings']}

### Recommendation Eligibility Check

Before generating recommendations, we ensure that the target
macronutrient distribution is reasonable and suitable for meal planning.

Extremely low macro targets can lead to invalid or impractical
recommendations.

This step validates whether the generated targets are eligible
for the recommendation system.

In [24]:
eligibility = nutrition.check_recommendation_eligibility(
    calories=macro_targets["Calories"],
    protein=macro_targets["Protein_g"],
    fat=macro_targets["Fat_g"],
    carbs=macro_targets["Carbs_g"]
)

eligibility

{'Eligible': True, 'Reasons': ['No Issues']}

### Nutrition-Based Recommendations

Using the computed macro targets, we generate recipe recommendations
that best match the nutritional requirements.

This step uses the nutrition-based scoring function from the
recommender module.

The recommendations are ranked based on how closely each recipe
matches the target calories and macronutrients.

In [25]:
nutrition_recommendations = recommender.generate_recommendations(
    df=recipes_df,
    calories=macro_targets["Calories"],
    protein=macro_targets["Protein_g"],
    carbs=macro_targets["Carbs_g"],
    fat=macro_targets["Fat_g"],
    meals_per_day=4,
    diet_type="Non-Vegetarian",
    top_n=10
)

nutrition_recommendations

,RecipeId,Name,BestServing,RecommendedCalories,RecommendedProtein,RecommendedCarbs,RecommendedFat,Distance,NutritionScore
0,262851.0,Prawn &amp; Chicken Orzo Paella,4,556.00,42.10,61.0,15.10,0.0116,0.9885
1,532395.0,Salt Cured Cod and Roasted Garlic Croutons,4,545.80,41.40,60.3,15.40,0.0128,0.9873
2,538802.0,5 - Way Chili,2,539.60,41.50,58.7,15.15,0.0141,0.9860
3,272242.0,Tex-Mex Turkey Tostadas,4,549.00,43.30,60.5,15.00,0.0141,0.9860
4,28862.0,Salmon Patty Pasta,4,553.20,41.70,61.0,15.00,0.0148,0.9853
5,392913.0,"Parmesan, Chicken and Broccoli Pasta",2,523.70,42.00,59.0,15.30,0.0188,0.9813
6,206944.0,"Rachael Ray Basil, Chili and Garlic Chicken",4,552.20,40.90,60.4,15.10,0.0193,0.9809
7,292510.0,Spring Dijon Chicken OAMC,6,533.10,42.50,60.8,14.10,0.0208,0.9794
8,467421.0,Chicken Parmesian,3,536.17,41.03,59.4,14.77,0.0224,0.9778
9,321618.0,Tuna Kofta,4,536.70,41.60,56.1,15.40,0.0237,0.9766


### Hybrid Recommendation

The validated daily nutrition targets are now passed to the hybrid
recommendation system.

The hybrid model combines the dynamic NutritionScore with the
precomputed ContentScore, PopularityScore, and SVDScore through
the previously defined StaticScore.

The result is a ranked set of recipes that balances nutritional
requirements with personalization and overall recipe quality.

In [26]:
hybrid_recommendations = generate_hybrid_recommendations(
    recipes_df=recipes_df,
    hybrid_static_df=hybrid_static_df,
    calories=macro_targets["Calories"],
    protein=macro_targets["Protein_g"],
    carbs=macro_targets["Carbs_g"],
    fat=macro_targets["Fat_g"],
    meals_per_day=4,
    diet_type="Non-Vegetarian",
    top_n=10
)

hybrid_recommendations

,RecipeId,Name,BestServing,RecommendedCalories,RecommendedProtein,RecommendedCarbs,RecommendedFat,Distance,NutritionScore,StaticScore,FinalScore
0,262851.0,Prawn &amp; Chicken Orzo Paella,4,556.00,42.10,61.00,15.10,0.0116,0.9885,0.370194,0.815019
1,95297.0,Creole Black-Eyed Peas,4,523.00,43.40,72.80,7.50,0.1303,0.8778,0.404140,0.799150
2,143519.0,Spinach Meatballs,4,526.90,36.50,59.00,15.40,0.0700,0.9324,0.379267,0.798847
3,108306.0,Pork Chops With Apples &amp; Stuffing,6,516.40,43.10,47.70,16.20,0.0639,0.9381,0.374501,0.796646
4,90772.0,"Black Beans, Chicken and Rice",4,566.10,37.90,61.60,18.10,0.0870,0.9166,0.382396,0.794866
5,64098.0,Spicy Chicken Linguine,4,538.90,44.90,52.70,16.10,0.0550,0.9465,0.368212,0.794137
6,10313.0,Mediterranean Chicken and Rice Pilaf,4,526.30,42.80,49.40,16.10,0.0503,0.9509,0.365999,0.793904
7,167741.0,"Pasta With Chicken, Broccoli, and Sun-Dried Tomatoes",4,577.50,41.10,60.10,15.20,0.0306,0.9699,0.351118,0.787573
8,147111.0,Ed &amp; Sandi's Pork Roast and Dumplings W/ Sauerkraut,6,534.45,45.68,52.28,14.77,0.0601,0.9417,0.363618,0.787383
9,229440.0,Cuban Pork Sandwiches,3,587.78,41.62,64.57,17.55,0.0658,0.9363,0.365796,0.787131


### Meal Planner Initialization

The daily nutritional targets are used to initialize the meal planner.

The planner maintains the remaining:
- Calories
- Protein
- Carbohydrates
- Fat
- Number of meals

It also keeps track of recipes already selected so that the same
recipe is not repeated within the daily meal plan.

In [27]:
# Initialize Daily Meal Planner

meals_per_day = 4

planner = recommender.initialize_meal_plan(
    calories=macro_targets["Calories"],
    protein=macro_targets["Protein_g"],
    carbs=macro_targets["Carbs_g"],
    fat=macro_targets["Fat_g"],
    meals_per_day=meals_per_day
)

planner

{'remaining_calories': 2181,
 'remaining_protein': 169.4,
 'remaining_carbs': 239.5,
 'remaining_fat': 60.6,
 'remaining_meals': 4,
 'selected_recipe_ids': [],
 'meals': []}

### Automatic Hybrid Meal Planner

The automatic meal planner integrates the hybrid recommendation system
with the sequential meal planning workflow.

For each meal, the planner:

1. Excludes recipes already selected during the day.
2. Generates hybrid recommendations using the remaining nutritional targets.
3. Automatically selects the highest-ranked recommendation.
4. Updates the remaining nutritional targets.
5. Repeats the process until all meals have been planned.

The hybrid score combines nutritional relevance with the precomputed
content, popularity, and collaborative filtering signals.

In [28]:
## Automatic Hybrid Meal Planner

def automatic_hybrid_meal_planner(
    recipes_df,
    hybrid_static_df,
    calories,
    protein,
    carbs,
    fat,
    meals_per_day,
    diet_type="Non-Vegetarian",
    top_n=10
):

    planner = recommender.initialize_meal_plan(
        calories,
        protein,
        carbs,
        fat,
        meals_per_day
    )

    for meal_number in range(meals_per_day):

        print("\n" + "=" * 70)
        print(f"Meal {meal_number + 1}")
        print("=" * 70)

        # Exclude previously selected recipes
        available_recipes = recipes_df[
            ~recipes_df["RecipeId"].isin(
                planner["selected_recipe_ids"]
            )
        ]

        # Generate hybrid recommendations
        recommendations = generate_hybrid_recommendations(
            recipes_df=available_recipes,
            hybrid_static_df=hybrid_static_df,
            calories=planner["remaining_calories"],
            protein=planner["remaining_protein"],
            carbs=planner["remaining_carbs"],
            fat=planner["remaining_fat"],
            meals_per_day=planner["remaining_meals"],
            diet_type=diet_type,
            top_n=top_n
        )

        # Automatically select highest-ranked recipe
        selected_recipe = recommendations.iloc[0].copy()

        print("\n✓ Selected Recipe")
        print("-" * 40)

        print(f"Name      : {selected_recipe['Name']}")
        print(f"Serving   : {selected_recipe['BestServing']}")

        print(
            f"Calories  : "
            f"{selected_recipe['RecommendedCalories']:.2f} kcal"
        )

        print(
            f"Protein   : "
            f"{selected_recipe['RecommendedProtein']:.2f} g"
        )

        print(
            f"Carbs     : "
            f"{selected_recipe['RecommendedCarbs']:.2f} g"
        )

        print(
            f"Fat       : "
            f"{selected_recipe['RecommendedFat']:.2f} g"
        )

        print(
            f"FinalScore: "
            f"{selected_recipe['FinalScore']:.4f}"
        )

        # Update planner state
        planner = recommender.update_meal_plan(
            planner,
            selected_recipe,
            meal_number
        )

        print("\nRemaining Nutrition Targets")
        print("-" * 40)

        print(
            f"Calories : "
            f"{planner['remaining_calories']:.2f} kcal"
        )

        print(
            f"Protein  : "
            f"{planner['remaining_protein']:.2f} g"
        )

        print(
            f"Carbs    : "
            f"{planner['remaining_carbs']:.2f} g"
        )

        print(
            f"Fat      : "
            f"{planner['remaining_fat']:.2f} g"
        )

    planner["meals"] = pd.DataFrame(
        planner["meals"]
    )

    print("\n" + "=" * 70)
    print("DAILY HYBRID MEAL PLAN GENERATED SUCCESSFULLY")
    print("=" * 70)

    return planner

In [29]:
meal_plan_auto = automatic_hybrid_meal_planner(
    recipes_df=recipes_df,
    hybrid_static_df=hybrid_static_df,
    calories=macro_targets["Calories"],
    protein=macro_targets["Protein_g"],
    carbs=macro_targets["Carbs_g"],
    fat=macro_targets["Fat_g"],
    meals_per_day=4,
    diet_type="Non-Vegetarian",
)


Meal 1

✓ Selected Recipe
----------------------------------------
Name      : Prawn &amp; Chicken Orzo Paella
Serving   : 4
Calories  : 556.00 kcal
Protein   : 42.10 g
Carbs     : 61.00 g
Fat       : 15.10 g
FinalScore: 0.8150

Remaining Nutrition Targets
----------------------------------------
Calories : 1625.00 kcal
Protein  : 127.30 g
Carbs    : 178.50 g
Fat      : 45.50 g

Meal 2

✓ Selected Recipe
----------------------------------------
Name      : Spinach Meatballs
Serving   : 4
Calories  : 526.90 kcal
Protein   : 36.50 g
Carbs     : 59.00 g
Fat       : 15.40 g
FinalScore: 0.7998

Remaining Nutrition Targets
----------------------------------------
Calories : 1098.10 kcal
Protein  : 90.80 g
Carbs    : 119.50 g
Fat      : 30.10 g

Meal 3

✓ Selected Recipe
----------------------------------------
Name      : Spicy Chicken Linguine
Serving   : 4
Calories  : 538.90 kcal
Protein   : 44.90 g
Carbs     : 52.70 g
Fat       : 16.10 g
FinalScore: 0.8014

Remaining Nutrition Targets
--

In [30]:
meal_plan_auto["meals"]

,RecipeId,Name,BestServing,RecommendedCalories,RecommendedProtein,RecommendedCarbs,RecommendedFat,Distance,NutritionScore,StaticScore,FinalScore,Meal
0,262851.0,Prawn &amp; Chicken Orzo Paella,4,556.0,42.1,61.0,15.1,0.0116,0.9885,0.370194,0.815019,1
1,143519.0,Spinach Meatballs,4,526.9,36.5,59.0,15.4,0.0677,0.9346,0.379267,0.799837,2
2,64098.0,Spicy Chicken Linguine,4,538.9,44.9,52.7,16.1,0.0381,0.9626,0.368212,0.801382,3
3,274344.0,New Year's Day Black-Eyed Peas,4,608.2,46.4,77.7,13.6,0.0594,0.9423,0.390984,0.815019,4


In [31]:
auto_summary = pd.DataFrame({
    "Nutrient": [
        "Calories",
        "Protein",
        "Carbohydrates",
        "Fat"
    ],
    "Target": [
        macro_targets["Calories"],
        macro_targets["Protein_g"],
        macro_targets["Carbs_g"],
        macro_targets["Fat_g"]
    ],
    "Consumed": [
        macro_targets["Calories"] -
        meal_plan_auto["remaining_calories"],

        macro_targets["Protein_g"] -
        meal_plan_auto["remaining_protein"],

        macro_targets["Carbs_g"] -
        meal_plan_auto["remaining_carbs"],

        macro_targets["Fat_g"] -
        meal_plan_auto["remaining_fat"]
    ],
    "Remaining": [
        meal_plan_auto["remaining_calories"],
        meal_plan_auto["remaining_protein"],
        meal_plan_auto["remaining_carbs"],
        meal_plan_auto["remaining_fat"]
    ]
})

auto_summary = auto_summary.round(2)

auto_summary

,Nutrient,Target,Consumed,Remaining
0,Calories,2181.0,2230.0,-49.0
1,Protein,169.4,169.9,-0.5
2,Carbohydrates,239.5,250.4,-10.9
3,Fat,60.6,60.2,0.4


### Interactive Hybrid Meal Planner

The interactive planner provides the user with the top hybrid
recommendations for each meal and allows a recipe to be selected.

After each selection:

- The selected recipe is added to the meal plan.
- Its nutritional contribution is deducted from the remaining targets.
- The selected recipe is excluded from subsequent meals.
- New hybrid recommendations are generated using the updated
  nutritional requirements.

This provides user control while retaining the same hybrid ranking
logic used by the automatic planner.

In [32]:
# Interactive Hybrid Meal Planner

def interactive_hybrid_meal_planner(
    recipes_df,
    hybrid_static_df,
    calories,
    protein,
    carbs,
    fat,
    meals_per_day,
    diet_type="Non-Vegetarian",
    top_n=10
):

    planner = recommender.initialize_meal_plan(
        calories,
        protein,
        carbs,
        fat,
        meals_per_day
    )

    for meal_number in range(meals_per_day):

        print("\n" + "=" * 70)
        print(f"MEAL {meal_number + 1}")
        print("=" * 70)

        # Exclude previously selected recipes
        available_recipes = recipes_df[
            ~recipes_df["RecipeId"].isin(
                planner["selected_recipe_ids"]
            )
        ]

        # Generate hybrid recommendations
        recommendations = generate_hybrid_recommendations(
            recipes_df=available_recipes,
            hybrid_static_df=hybrid_static_df,
            calories=planner["remaining_calories"],
            protein=planner["remaining_protein"],
            carbs=planner["remaining_carbs"],
            fat=planner["remaining_fat"],
            meals_per_day=planner["remaining_meals"],
            diet_type=diet_type,
            top_n=top_n
        )

        # Display recommendations
        display_columns = [
            "RecipeId",
            "Name",
            "BestServing",
            "RecommendedCalories",
            "RecommendedProtein",
            "RecommendedCarbs",
            "RecommendedFat",
            "NutritionScore",
            "StaticScore",
            "FinalScore"
        ]

        display(
            recommendations[
                display_columns
            ].reset_index(drop=True)
        )

        # User selection
        while True:

            try:
                choice = int(
                    input(
                        f"\nChoose a recipe (1-{len(recommendations)}): "
                    )
                )

                if 1 <= choice <= len(recommendations):
                    break

                print(
                    f"Please enter a number between "
                    f"1 and {len(recommendations)}."
                )

            except ValueError:

                print(
                    "Invalid input. Please enter a number."
                )

        # Convert user choice to DataFrame index
        selected_recipe = recommendations.iloc[
            choice - 1
        ].copy()

        # Update planner
        planner = recommender.update_meal_plan(
            planner,
            selected_recipe,
            meal_number
        )

        # Display selected recipe
        print("\n✓ Selected Recipe")
        print("-" * 40)
        print(f"Name     : {selected_recipe['Name']}")
        print(f"Serving  : {selected_recipe['BestServing']}")
        print(
            f"Calories : "
            f"{selected_recipe['RecommendedCalories']:.2f} kcal"
        )
        print(
            f"Protein  : "
            f"{selected_recipe['RecommendedProtein']:.2f} g"
        )
        print(
            f"Carbs    : "
            f"{selected_recipe['RecommendedCarbs']:.2f} g"
        )
        print(
            f"Fat      : "
            f"{selected_recipe['RecommendedFat']:.2f} g"
        )

        print("\nRemaining Nutrition Targets")
        print("-" * 40)
        print(
            f"Calories : "
            f"{planner['remaining_calories']:.2f} kcal"
        )
        print(
            f"Protein  : "
            f"{planner['remaining_protein']:.2f} g"
        )
        print(
            f"Carbs    : "
            f"{planner['remaining_carbs']:.2f} g"
        )
        print(
            f"Fat      : "
            f"{planner['remaining_fat']:.2f} g"
        )

    planner["meals"] = pd.DataFrame(
        planner["meals"]
    )

    print("\n" + "=" * 70)
    print("DAILY HYBRID MEAL PLAN GENERATED SUCCESSFULLY")
    print("=" * 70)

    return planner

In [33]:
meal_plan_interactive = interactive_hybrid_meal_planner(
    recipes_df=recipes_df,
    hybrid_static_df=hybrid_static_df,
    calories=macro_targets["Calories"],
    protein=macro_targets["Protein_g"],
    carbs=macro_targets["Carbs_g"],
    fat=macro_targets["Fat_g"],
    meals_per_day=4,
    diet_type="Non-Vegetarian",
    top_n=10
)


MEAL 1


,RecipeId,Name,BestServing,RecommendedCalories,RecommendedProtein,RecommendedCarbs,RecommendedFat,NutritionScore,StaticScore,FinalScore
0,262851.0,Prawn &amp; Chicken Orzo Paella,4,556.00,42.10,61.00,15.10,0.9885,0.370194,0.815019
1,95297.0,Creole Black-Eyed Peas,4,523.00,43.40,72.80,7.50,0.8778,0.404140,0.799150
2,143519.0,Spinach Meatballs,4,526.90,36.50,59.00,15.40,0.9324,0.379267,0.798847
3,108306.0,Pork Chops With Apples &amp; Stuffing,6,516.40,43.10,47.70,16.20,0.9381,0.374501,0.796646
4,90772.0,"Black Beans, Chicken and Rice",4,566.10,37.90,61.60,18.10,0.9166,0.382396,0.794866
5,64098.0,Spicy Chicken Linguine,4,538.90,44.90,52.70,16.10,0.9465,0.368212,0.794137
6,10313.0,Mediterranean Chicken and Rice Pilaf,4,526.30,42.80,49.40,16.10,0.9509,0.365999,0.793904
7,167741.0,"Pasta With Chicken, Broccoli, and Sun-Dried Tomatoes",4,577.50,41.10,60.10,15.20,0.9699,0.351118,0.787573
8,147111.0,Ed &amp; Sandi's Pork Roast and Dumplings W/ Sauerkraut,6,534.45,45.68,52.28,14.77,0.9417,0.363618,0.787383
9,229440.0,Cuban Pork Sandwiches,3,587.78,41.62,64.57,17.55,0.9363,0.365796,0.787131



✓ Selected Recipe
----------------------------------------
Name     : Spinach Meatballs
Serving  : 4
Calories : 526.90 kcal
Protein  : 36.50 g
Carbs    : 59.00 g
Fat      : 15.40 g

Remaining Nutrition Targets
----------------------------------------
Calories : 1654.10 kcal
Protein  : 132.90 g
Carbs    : 180.50 g
Fat      : 45.20 g

MEAL 2


,RecipeId,Name,BestServing,RecommendedCalories,RecommendedProtein,RecommendedCarbs,RecommendedFat,NutritionScore,StaticScore,FinalScore
0,262851.0,Prawn &amp; Chicken Orzo Paella,4,556.00,42.10,61.00,15.10,0.9755,0.370194,0.809169
1,64098.0,Spicy Chicken Linguine,4,538.90,44.90,52.70,16.10,0.9597,0.368212,0.800077
2,95297.0,Creole Black-Eyed Peas,4,523.00,43.40,72.80,7.50,0.8778,0.404140,0.799150
3,52655.0,White Chili,9,562.05,45.63,58.41,16.74,0.9619,0.362653,0.795508
4,274344.0,New Year's Day Black-Eyed Peas,4,608.20,46.40,77.70,13.60,0.8974,0.390984,0.794814
5,147111.0,Ed &amp; Sandi's Pork Roast and Dumplings W/ Sauerkraut,6,534.45,45.68,52.28,14.77,0.9568,0.363618,0.794178
6,108306.0,Pork Chops With Apples &amp; Stuffing,6,516.40,43.10,47.70,16.20,0.9303,0.374501,0.793136
7,231007.0,Chicken Piccata With Mushrooms,4,565.60,45.80,52.80,18.10,0.9325,0.371935,0.791560
8,90772.0,"Black Beans, Chicken and Rice",4,566.10,37.90,61.60,18.10,0.9052,0.382396,0.789736
9,10313.0,Mediterranean Chicken and Rice Pilaf,4,526.30,42.80,49.40,16.10,0.9377,0.365999,0.787964



✓ Selected Recipe
----------------------------------------
Name     : White Chili
Serving  : 9
Calories : 562.05 kcal
Protein  : 45.63 g
Carbs    : 58.41 g
Fat      : 16.74 g

Remaining Nutrition Targets
----------------------------------------
Calories : 1092.05 kcal
Protein  : 87.27 g
Carbs    : 122.09 g
Fat      : 28.46 g

MEAL 3


,RecipeId,Name,BestServing,RecommendedCalories,RecommendedProtein,RecommendedCarbs,RecommendedFat,NutritionScore,StaticScore,FinalScore
0,262851.0,Prawn &amp; Chicken Orzo Paella,4,556.00,42.10,61.00,15.10,0.9716,0.370194,0.807414
1,95297.0,Creole Black-Eyed Peas,4,523.00,43.40,72.80,7.50,0.8917,0.404140,0.805405
2,274344.0,New Year's Day Black-Eyed Peas,4,608.20,46.40,77.70,13.60,0.8985,0.390984,0.795309
3,64098.0,Spicy Chicken Linguine,4,538.90,44.90,52.70,16.10,0.9458,0.368212,0.793822
4,108306.0,Pork Chops With Apples &amp; Stuffing,6,516.40,43.10,47.70,16.20,0.9280,0.374501,0.792101
5,147111.0,Ed &amp; Sandi's Pork Roast and Dumplings W/ Sauerkraut,6,534.45,45.68,52.28,14.77,0.9490,0.363618,0.790668
6,90772.0,"Black Beans, Chicken and Rice",4,566.10,37.90,61.60,18.10,0.8996,0.382396,0.787216
7,10313.0,Mediterranean Chicken and Rice Pilaf,4,526.30,42.80,49.40,16.10,0.9354,0.365999,0.786929
8,112849.0,Pancit Bihon (or Bijon),5,544.42,44.25,57.50,13.42,0.9765,0.347309,0.786734
9,89852.0,Chicken Pesto Pasta,4,536.90,43.90,59.20,13.00,0.9754,0.344995,0.783925



✓ Selected Recipe
----------------------------------------
Name     : Ed &amp; Sandi's Pork Roast and Dumplings W/ Sauerkraut
Serving  : 6
Calories : 534.45 kcal
Protein  : 45.68 g
Carbs    : 52.28 g
Fat      : 14.77 g

Remaining Nutrition Targets
----------------------------------------
Calories : 557.60 kcal
Protein  : 41.59 g
Carbs    : 69.81 g
Fat      : 13.69 g

MEAL 4


,RecipeId,Name,BestServing,RecommendedCalories,RecommendedProtein,RecommendedCarbs,RecommendedFat,NutritionScore,StaticScore,FinalScore
0,95297.0,Creole Black-Eyed Peas,4,523.00,43.40,72.80,7.50,0.8956,0.404140,0.807160
1,262851.0,Prawn &amp; Chicken Orzo Paella,4,556.00,42.10,61.00,15.10,0.9607,0.370194,0.802509
2,274344.0,New Year's Day Black-Eyed Peas,4,608.20,46.40,77.70,13.60,0.9126,0.390984,0.801654
3,90772.0,"Black Beans, Chicken and Rice",4,566.10,37.90,61.60,18.10,0.8994,0.382396,0.787126
4,38244.0,Oriental Flank Steak with Asparagus and Wild Rice Pilaf,4,545.60,39.70,67.80,12.90,0.9631,0.353267,0.786662
5,229440.0,Cuban Pork Sandwiches,3,587.78,41.62,64.57,17.55,0.9323,0.365796,0.785331
6,191978.0,Chicken and Broccoli Pasta With Pesto,4,587.10,40.10,81.80,13.20,0.9405,0.358297,0.781522
7,359976.0,Chicken and Rice With Spring Vegetables,4,599.70,41.10,68.10,16.90,0.9359,0.359136,0.780291
8,48461.0,white chili,6,571.00,39.10,71.40,16.00,0.9419,0.356282,0.780137
9,205922.0,Chicken Parmesan,3,499.42,38.40,53.40,13.73,0.9070,0.371340,0.779490



✓ Selected Recipe
----------------------------------------
Name     : Creole Black-Eyed Peas
Serving  : 4
Calories : 523.00 kcal
Protein  : 43.40 g
Carbs    : 72.80 g
Fat      : 7.50 g

Remaining Nutrition Targets
----------------------------------------
Calories : 34.60 kcal
Protein  : -1.81 g
Carbs    : -2.99 g
Fat      : 6.19 g

DAILY HYBRID MEAL PLAN GENERATED SUCCESSFULLY


In [34]:
# Final Interactive Meal Plan

meal_plan_interactive["meals"]

,RecipeId,Name,BestServing,RecommendedCalories,RecommendedProtein,RecommendedCarbs,RecommendedFat,Distance,NutritionScore,StaticScore,FinalScore,Meal
0,143519.0,Spinach Meatballs,4,526.90,36.50,59.00,15.40,0.0700,0.9324,0.379267,0.798847,1
1,52655.0,White Chili,9,562.05,45.63,58.41,16.74,0.0389,0.9619,0.362653,0.795508,2
2,147111.0,Ed &amp; Sandi's Pork Roast and Dumplings W/ Sauerkraut,6,534.45,45.68,52.28,14.77,0.0524,0.9490,0.363618,0.790668,3
3,95297.0,Creole Black-Eyed Peas,4,523.00,43.40,72.80,7.50,0.1103,0.8956,0.404140,0.807160,4


In [35]:
interactive_summary = pd.DataFrame({
    "Nutrient": [
        "Calories",
        "Protein",
        "Carbohydrates",
        "Fat"
    ],
    "Target": [
        macro_targets["Calories"],
        macro_targets["Protein_g"],
        macro_targets["Carbs_g"],
        macro_targets["Fat_g"]
    ],
    "Consumed": [
        macro_targets["Calories"] -
        meal_plan_interactive["remaining_calories"],

        macro_targets["Protein_g"] -
        meal_plan_interactive["remaining_protein"],

        macro_targets["Carbs_g"] -
        meal_plan_interactive["remaining_carbs"],

        macro_targets["Fat_g"] -
        meal_plan_interactive["remaining_fat"]
    ],
    "Remaining": [
        meal_plan_interactive["remaining_calories"],
        meal_plan_interactive["remaining_protein"],
        meal_plan_interactive["remaining_carbs"],
        meal_plan_interactive["remaining_fat"]
    ]
})

interactive_summary = interactive_summary.round(2)

interactive_summary

,Nutrient,Target,Consumed,Remaining
0,Calories,2181.0,2146.40,34.60
1,Protein,169.4,171.21,-1.81
2,Carbohydrates,239.5,242.49,-2.99
3,Fat,60.6,54.41,6.19
